# Running ETL to Build the Document Corpus

This notebook walks through the process for setting up the corpus of Full Stack documents that the bot searches over.

In each case, we have to
- Extract data from its natural habitat, like YouTube or GitHub
- Transform it into a format that is useful for our purposes
- Load it into our database in that format

hence the acronym "ETL".

In [11]:
#!make secrets  # you'll need credentials for Mongo and Modal to run this
db, collection = "fsdl-dev", "ask-fsdl"  # we run this in a dev database

# drop the collection if it exists, it's just dev
#!modal run app.py::drop_docs --db {db} --collection {collection}
#!modal run app.py::drop_docs --db fsdl-dev --collection ask-fsdl

In [15]:
import json
from pathlib import Path
import pprint

from dotenv import load_dotenv
load_dotenv(".env.dev")  # 加载 MongoDB、Modal 等本地配置

from etl import markdown, pdfs, shared, videos
from etl.shared import display_modal_image

pp = pprint.PrettyPrinter(indent=2)

## PDFs: arXiV Papers

```bash
!modal run etl/pdfs.py --json-path data/llm-papers.json
```

In [ ]:
display_modal_image(shared.image)

In [ ]:
display_modal_image(pdfs.image)

In [2]:
papers_path = Path("data") / "llm-papers.json"

with open(papers_path) as f:
    pdf_infos = json.load(f)

pdf_infos[:100:20]

[{'tags': ['Multimodal', 'Vision', 'Internals'],
  'title': 'On the Hidden Mystery of OCR in Large Multimodal Models',
  'url': 'https://arxiv.org/abs/2305.07895'},
 {'tags': ['Evaluation'],
  'title': 'Asking Crowdworkers to Write Entailment Examples: The Best of Bad Options',
  'url': 'https://aclanthology.org/2020.aacl-main.68/'},
 {'tags': ['Prompting', 'Critical', 'Philosophy', 'Simulation'],
  'title': 'Inducing anxiety in large language models increases exploration and bias',
  'url': 'https://arxiv.org/abs/2304.11111v1'},
 {'tags': ['Ahead of Its Time'],
  'title': 'PROGRAMS WITH COMMON SENSE',
  'url': 'http://jmc.stanford.edu/articles/mcc59/mcc59.pdf'},
 {'tags': ['Ahead of Its Time', 'External Memory', 'Reasoning', 'Philosophy'],
  'title': 'Heuristic Problem Solving By Computer',
  'url': 'https://iiif.library.cmu.edu/file/Simon_box00065_fld04954_bdl0001_doc0001/Simon_box00065_fld04954_bdl0001_doc0001.pdf'}]

In [3]:
async with pdfs.app.run():
    # first, we enrich the paper data by finding direct PDF URLs where we can
    paper_data = [p async for p in pdfs.get_pdf_url.map.aio(
        pdf_infos[::25], #subsampling to run faster
        return_exceptions=True
    )]
    # then we turn the PDFs into JSON documents
    results = [r async for r in pdfs.extract_pdf.map.aio(  # each pdf creates a list of documents, one per page
        paper_data, return_exceptions=True
    )]
    documents = shared.unchunk(results)

In [4]:
pp.pprint(documents[0]["metadata"])

{ 'arxiv_id': '2305.07895',
  'date': datetime.datetime(2024, 8, 26, 2, 37, 14, tzinfo=datetime.timezone.utc),
  'full-title': 'OCRBench: On the Hidden Mystery of OCR in Large Multimodal '
                'Models - p0',
  'ignore': False,
  'is_endmatter': False,
  'page': 0,
  'sha256': '36ef214dedbe828a942abddd00eb3887ba7b78afa3b4640b389544260e8cf84b',
  'source': 'https://arxiv.org/abs/2305.07895',
  'title': 'OCRBench: On the Hidden Mystery of OCR in Large Multimodal Models'}


In [8]:
pp.pprint(len(documents[0]['text']))

3646


In [26]:
from IPython.display import IFrame

IFrame(src=documents[0]["metadata"]["source"], width=800, height=400)

In [12]:
async with shared.app.run():
    # we split our document list into 10 pieces, so that we don't open too many connections
    chunked_documents = list(shared.chunk_into(documents, 10))
    results = [r async for r in shared.add_to_document_db.map.aio(chunked_documents, kwargs={"db": db, "collection": collection})]

In [16]:
async with shared.app.run():
    import docstore
    # pull only arxiv papers
    query = { "metadata.source": { "$regex": "arxiv\.org", "$options": "i" } }
    # project out the text field, it can get large
    projection = {"text": 0}
    # get just one result to show it worked
    result = docstore.query_one(query, projection, db=db, collection=collection)

pp.pprint(result)

{ '_id': ObjectId('69cbf07f4bc5167be9ca7520'),
  'metadata': { 'arxiv_id': '2305.07895',
                'date': datetime.datetime(2024, 8, 26, 2, 37, 14),
                'full-title': 'OCRBench: On the Hidden Mystery of OCR in Large '
                              'Multimodal Models - p0',
                'ignore': False,
                'is_endmatter': False,
                'page': 0,
                'sha256': '36ef214dedbe828a942abddd00eb3887ba7b78afa3b4640b389544260e8cf84b',
                'source': 'https://arxiv.org/abs/2305.07895',
                'title': 'OCRBench: On the Hidden Mystery of OCR in Large '
                         'Multimodal Models'},
  'type': 'Document'}


## Markdown Files: Lectures

```bash
!modal run etl/markdown.py --json-path data/lectures-2022.json
```

In [ ]:
display_modal_image(markdown.image)

In [17]:
markdown_path = Path("data") / "lectures-2022.json"

with open(markdown_path) as f:
  markdown_corpus = json.load(f)

website_url, md_url = (
  markdown_corpus["website_url_base"],
  markdown_corpus["md_url_base"],
)

lectures = markdown_corpus["lectures"]

lectures[0]

{'slug': 'lecture-1-course-vision-and-when-to-use-ml',
 'title': 'Course Vision & When to Use ML'}

In [22]:
async with markdown.app.run():
    results = [r async for r in markdown.to_documents.map.aio(
        lectures,
        kwargs={"website_url": website_url, "md_url": md_url},
        return_exceptions=True,
    )]
    documents = shared.unchunk(results)

In [23]:
pp.pprint(documents[1]["metadata"])

{ 'full-title': 'Course Vision & When to Use ML - 1 - Course Vision',
  'heading': '1 - Course Vision',
  'ignore': False,
  'sha256': '9d8ff6ab9eec7206029c3d575385e8e8877ff099f81834df2b856d70e3b14890',
  'source': 'https://fullstackdeeplearning.com/course/2022/lecture-1-course-vision-and-when-to-use-ml#1-course-vision',
  'title': 'Course Vision & When to Use ML'}


In [24]:
from IPython.display import IFrame

IFrame(src=documents[1]["metadata"]["source"], width=800, height=400)

In [27]:
async with shared.app.run():
    chunked_documents = list(shared.chunk_into(documents, 10))
    results = [r async for r in shared.add_to_document_db.map.aio(chunked_documents, kwargs={"db": db, "collection": collection})]

In [28]:
async with shared.app.run():
    import docstore
    # pull only lectures
    query = { "metadata.source": { "$regex": "lecture", "$options": "i" } }
    # project out the text field, it can get large
    projection = {"text": 0}
    # get just one result to show it worked
    result = docstore.query_one(query, projection, db=db, collection=collection)

pp.pprint(result)

{ '_id': ObjectId('69cbf67aae90b4a898cf5c6e'),
  'metadata': { 'full-title': 'Course Vision & When to Use ML - ',
                'heading': '',
                'ignore': False,
                'sha256': 'd2ae0baaf6bd6e1caa991c6ec89e437dc2497b9ec48a26b4f81a09320b53dd00',
                'source': 'https://fullstackdeeplearning.com/course/2022/lecture-1-course-vision-and-when-to-use-ml#',
                'title': 'Course Vision & When to Use ML'}}


## Videos: YouTube Transcripts

> 视频禁用了字幕抓取，youtube-transcript-api不可用
> 
> yt-dlp方案要做机器人检测，无头浏览器做不了

In [ ]:
!modal run etl/videos.py --json-path data/videos.json

In [ ]:
display_modal_image(videos.image)

In [29]:
videos_path = Path("data") / "videos.json"

with open(videos_path) as f:
    video_infos = json.load(f)

video_ids = [video["id"] for video in video_infos]

video_infos[0]

{'id': '-Iob-FW5jVM',
 'title': 'Lecture 01: When to Use ML and Course Vision (FSDL 2022)'}

In [30]:
async with videos.app.run():
    results = [r async for r in videos.extract_subtitles.map.aio(
        video_infos[-3:],  # subsampling to run faster
        return_exceptions=True,
    )]
    documents = shared.unchunk(results)

TypeError: 'ExecutionError' object is not iterable

In [ ]:
pp.pprint(documents[1]["metadata"])

In [ ]:
from IPython.display import YouTubeVideo

id_str, time_str = documents[1]["metadata"]["source"].split("?v=")[-1].split("&t=")
YouTubeVideo(id_str, start=int(time_str.strip("s")), width=800, height=400)

In [ ]:
async with shared.app.run():
    chunked_documents = list(shared.chunk_into(documents, 10))
    results = [r async for r in shared.add_to_document_db.map.aio(chunked_documents, kwargs={"db": db, "collection": collection})]

In [ ]:
async with shared.app.run():
    import docstore
    # pull only lectures
    query = { "metadata.source": { "$regex": "youtube", "$options": "i" } }
    # project out the text field, it can get large
    projection = {"text": 0}
    # get just one result to show it worked
    result = docstore.query_one(query, projection, db=db, collection=collection)

pp.pprint(result)